# Data Preprocessing

## Step 1

In [ ]:
import rioxarray as rxr
import geopandas as gpd

## prodes = rxr.open_rasterio('../inputs/PRODES/prodes_brasil_2024.tif', masked=True)

In [ ]:
prodes = rxr.open_rasterio('../inputs/PRODES/prodes_brasil_2024.tif', masked=True)
print(prodes.rio.crs)

## para = gpd.read_file('../inputs/para_boundary/para.gpkg')

In [ ]:
para = gpd.read_file('../inputs/para_boundary/para.gpkg')
print(para.crs)

## !gdalwarp \

In [ ]:
!gdalwarp \
-t_srs EPSG:31982 \
-cutline ../inputs/para_boundary/para.gpkg \
-crop_to_cutline \
-dstnodata -9999 \
-ot Int16 \
../inputs/PRODES/prodes_brasil_2024.tif \
../inputs/label/prodes_para.tif

## !gdal_calc.py \

In [ ]:
!gdal_calc.py \
-A ../inputs/label/prodes_para.tif \
--outfile=../inputs/label/para_two_class.tif \
--calc="(A>=0)*(A<=64)" \
--type=Int16 \
--NoDataValue=-9999 \
--quiet

## !ls ../inputs/landsat_rgb_qa/*.tif | sort | \

In [ ]:
!ls ../inputs/landsat_rgb_qa/*.tif | sort | \
gdalbuildvrt \
-b 7 \
-input_file_list /vsistdin/ \
qa_cloud.vrt

## %%bash

In [ ]:
%%bash
gdalwarp \
  -t_srs EPSG:31982 \
  -r near \
  -cutline ../inputs/para_boundary/para.gpkg \
  -crop_to_cutline \
  -te $(gdalinfo -json ../inputs/label/para_two_class.tif | python -c "import json,sys; j=json.load(sys.stdin); cc=j['cornerCoordinates']; xmin,ymax=cc['upperLeft']; xmax,ymin=cc['lowerRight']; print(xmin,ymin,xmax,ymax)") \
  -ts $(gdalinfo -json ../inputs/label/para_two_class.tif | python -c "import json,sys; j=json.load(sys.stdin); print(j['size'][0], j['size'][1])") \
  -ot Int16 \
  -dstnodata -9999 \
  qa_cloud.vrt \
  qa_crop.vrt

## !gdal_calc.py \

In [ ]:
!gdal_calc.py \
-A ../inputs/label/para_two_class.tif \
-B qa_crop.vrt \
--outfile=../inputs/label/para_three_class.tif \
--calc="where(B==1, 2, A)" \
--type=Int16 \
--NoDataValue=-9999 \
--quiet

## !ls ../inputs/landsat_rgb_qa/*.tif | sort | \

In [ ]:
!ls ../inputs/landsat_rgb_qa/*.tif | sort | \
gdalbuildvrt \
-b 1 -b 2 -b 3 -b 4 -b 5 -b 6\
-input_file_list /vsistdin/ \
rgb.vrt

## %%bash

In [ ]:
%%bash

gdalwarp \
  -of GTiff \
  -t_srs EPSG:31982 \
  -r bilinear \
  -cutline ../inputs/para_boundary/para.gpkg \
  -crop_to_cutline \
  -te $(gdalinfo -json ../inputs/label/para_three_class.tif | python -c "import json,sys; j=json.load(sys.stdin); cc=j['cornerCoordinates']; xmin,ymax=cc['upperLeft']; xmax,ymin=cc['lowerRight']; print(xmin,ymin,xmax,ymax)") \
  -ts $(gdalinfo -json ../inputs/label/para_three_class.tif | python -c "import json,sys; j=json.load(sys.stdin); print(j['size'][0], j['size'][1])") \
  -ot Int16 \
  -dstnodata -9999 \
  rgb.vrt \
  ../inputs/map/map.tif